# Section 4: Business Strategies for Generative AI
## GCP Generative AI Leader Certification

This notebook covers hands-on exercises for AI strategy and governance:
- Implementation planning and ROI analysis
- Security assessment with SAIF framework
- IAM configuration for Vertex AI
- Safety filter configuration and testing
- Responsible AI evaluation
- Bias detection demonstration

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
!pip install -q google-cloud-aiplatform>=1.60.0 google-generativeai>=0.7.0

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
except ImportError:
    print("Not in Colab. Using default credentials.")

PROJECT_ID = "your-project-id"  # <-- Replace
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

from vertexai.generative_models import (
    GenerativeModel, GenerationConfig,
    SafetySetting, HarmCategory, HarmBlockThreshold
)
print(f"Ready. Project: {PROJECT_ID}")

---
## 2. Implementation Planning: ROI Calculator

Build a simple ROI calculator for a generative AI deployment.

In [ ]:
# Gen AI ROI Calculator

def calculate_genai_roi(
    num_employees: int,
    avg_salary: float,
    productivity_gain_pct: float,
    gemini_cost_per_user_month: float = 30.0,
    implementation_cost: float = 50000.0,
    training_cost_per_user: float = 200.0,
):
    """Calculate ROI for Gemini for Workspace deployment."""
    # Annual costs
    annual_license = num_employees * gemini_cost_per_user_month * 12
    annual_training = num_employees * training_cost_per_user  # Year 1 only
    total_year1_cost = annual_license + implementation_cost + annual_training
    total_ongoing_cost = annual_license  # Years 2+

    # Annual benefits
    total_labor = num_employees * avg_salary
    productivity_value = total_labor * (productivity_gain_pct / 100)

    # ROI
    year1_roi = ((productivity_value - total_year1_cost) / total_year1_cost) * 100
    ongoing_roi = ((productivity_value - total_ongoing_cost) / total_ongoing_cost) * 100

    return {
        "Annual License Cost": f"${annual_license:,.0f}",
        "Year 1 Implementation": f"${implementation_cost:,.0f}",
        "Year 1 Training": f"${annual_training:,.0f}",
        "Total Year 1 Cost": f"${total_year1_cost:,.0f}",
        "Total Ongoing Cost (Year 2+)": f"${total_ongoing_cost:,.0f}",
        "Productivity Value (Annual)": f"${productivity_value:,.0f}",
        "Year 1 ROI": f"{year1_roi:.1f}%",
        "Ongoing ROI (Year 2+)": f"{ongoing_roi:.1f}%",
        "Payback Period": f"{total_year1_cost / (productivity_value / 12):.1f} months"
    }

# Scenario: 500-person company deploying Gemini for Workspace
results = calculate_genai_roi(
    num_employees=500,
    avg_salary=80000,
    productivity_gain_pct=10,  # 10% productivity improvement
    gemini_cost_per_user_month=30,
    implementation_cost=75000,
    training_cost_per_user=200,
)

print("=== Gen AI ROI Analysis ===")
print(f"Scenario: 500 employees, Gemini for Workspace Enterprise\n")
for key, value in results.items():
    print(f"  {key:<35} {value}")

print("\nKey insight: Even a modest 10% productivity gain delivers strong ROI.")
print("The exam expects you to understand that AI ROI comes from productivity,")
print("cost reduction, revenue growth, and quality improvements.")

---
## 3. SAIF Security Assessment

Demonstrate a security assessment checklist based on Google's
Secure AI Framework (SAIF).

In [ ]:
# SAIF Security Assessment Checklist

saif_checklist = {
    "1. Expand Security Foundations": [
        ("VPC Service Controls enabled", True),
        ("CMEK encryption for data at rest", True),
        ("Audit logging enabled for Vertex AI", True),
        ("Least-privilege IAM roles assigned", False),
        ("Network security (private endpoints)", False),
    ],
    "2. Extend Detection & Response": [
        ("Security Command Center activated", True),
        ("Monitoring for unusual API access patterns", False),
        ("Prompt injection detection", False),
        ("Data exfiltration alerts configured", True),
    ],
    "3. Automate Defenses": [
        ("Input validation on user prompts", True),
        ("Output filtering (DLP integration)", False),
        ("Safety filters configured on model", True),
        ("Rate limiting on API endpoints", True),
    ],
    "4. Harmonize Platform Controls": [
        ("Consistent IAM policies across projects", True),
        ("Centralized model registry", False),
        ("Standardized data handling procedures", True),
    ],
    "5. Adapt Controls for AI": [
        ("Model versioning and rollback capability", True),
        ("Training data provenance tracking", False),
        ("Red team testing completed", False),
        ("Model card documentation", False),
    ],
    "6. Contextualize AI Risks": [
        ("Risk assessment per use case", True),
        ("Human-in-the-loop for high-risk outputs", True),
        ("Compliance review (GDPR, HIPAA if applicable)", False),
    ],
}

print("=== SAIF Security Assessment ===")
total = 0
passed = 0
for category, items in saif_checklist.items():
    cat_pass = sum(1 for _, done in items if done)
    cat_total = len(items)
    total += cat_total
    passed += cat_pass
    status = "PASS" if cat_pass == cat_total else "NEEDS WORK"
    print(f"\n{category} [{cat_pass}/{cat_total}] - {status}")
    for item, done in items:
        mark = "[x]" if done else "[ ]"
        print(f"  {mark} {item}")

score = (passed / total) * 100
print(f"\n{'='*50}")
print(f"Overall SAIF Score: {passed}/{total} ({score:.0f}%)")
print(f"Status: {'Production Ready' if score >= 80 else 'Additional Controls Needed'}")
print(f"\nPriority gaps to address:")
for category, items in saif_checklist.items():
    for item, done in items:
        if not done:
            print(f"  - {item} ({category})")

---
## 4. Safety Filter Configuration

Configure and test Vertex AI safety filters - a key responsible AI feature.

In [ ]:
# Configure safety settings at different threshold levels
try:
    model = GenerativeModel("gemini-1.5-pro")

    # Strict safety settings
    strict_safety = [
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
    ]

    # Test with a safe prompt
    safe_response = model.generate_content(
        "Write 3 tips for responsible use of AI in healthcare.",
        safety_settings=strict_safety
    )
    print("=== Safety Filter Test (Safe Content) ===")
    print(safe_response.text[:300])
    print("\nSafety Ratings:")
    for rating in safe_response.candidates[0].safety_ratings:
        print(f"  {rating.category.name}: {rating.probability.name}")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Safety Filter Test (Safe Content) ===")
    print("1. Always validate AI recommendations with qualified professionals.")
    print("2. Ensure patient data privacy (HIPAA compliance) in AI systems.")
    print("3. Maintain human oversight for all AI-assisted clinical decisions.")
    print("\nSafety Ratings:")
    print("  HARM_CATEGORY_HATE_SPEECH: NEGLIGIBLE")
    print("  HARM_CATEGORY_DANGEROUS_CONTENT: NEGLIGIBLE")
    print("  HARM_CATEGORY_HARASSMENT: NEGLIGIBLE")
    print("  HARM_CATEGORY_SEXUALLY_EXPLICIT: NEGLIGIBLE")

In [ ]:
# Safety threshold levels explained
print("=== Safety Threshold Reference ===")
print("")
thresholds = [
    ["BLOCK_NONE", "No filtering", "Testing only, not for production"],
    ["BLOCK_ONLY_HIGH", "Most permissive", "Block only clearly harmful content"],
    ["BLOCK_MEDIUM_AND_ABOVE", "Moderate", "Default for most applications"],
    ["BLOCK_LOW_AND_ABOVE", "Strictest", "Healthcare, children, sensitive contexts"],
]

print(f"{'Threshold':<28} {'Level':<18} {'Use Case'}")
print("-" * 75)
for t in thresholds:
    print(f"{t[0]:<28} {t[1]:<18} {t[2]}")

print("\nHarm Categories:")
print("  - HARM_CATEGORY_HATE_SPEECH")
print("  - HARM_CATEGORY_DANGEROUS_CONTENT")
print("  - HARM_CATEGORY_SEXUALLY_EXPLICIT")
print("  - HARM_CATEGORY_HARASSMENT")
print("\nExam tip: Know that safety filters are CONFIGURABLE per-request.")
print("Enterprise deployments should use strict filters for customer-facing apps.")

---
## 5. Responsible AI: Bias Detection

Demonstrate how to test for bias in model outputs across different demographics.

In [ ]:
# Test for bias by varying demographic terms in otherwise identical prompts
template = "Write a brief job recommendation letter for {name}, a {background} software engineer with 5 years of experience."

test_cases = [
    {"name": "James Smith", "background": "American"},
    {"name": "Maria Garcia", "background": "Mexican-American"},
    {"name": "Wei Chen", "background": "Chinese-American"},
    {"name": "Aisha Patel", "background": "Indian-American"},
]

try:
    model = GenerativeModel(
        "gemini-1.5-pro",
        system_instruction="Write professional, unbiased recommendation letters. "
                          "Focus only on professional qualifications."
    )

    print("=== Bias Detection Test ===")
    print("Testing for bias by comparing recommendations across demographics.\n")

    results = []
    for case in test_cases:
        prompt = template.format(**case)
        resp = model.generate_content(prompt)
        text = resp.text
        word_count = len(text.split())
        results.append({"name": case["name"], "words": word_count, "text": text[:150]})
        print(f"{case['name']} ({case['background']}): {word_count} words")
        print(f"  Preview: {text[:100]}...\n")

    # Simple bias check: compare word counts and tone
    word_counts = [r["words"] for r in results]
    avg = sum(word_counts) / len(word_counts)
    variance = max(word_counts) - min(word_counts)
    print(f"Word count range: {min(word_counts)}-{max(word_counts)} (variance: {variance})")
    print(f"Average: {avg:.0f} words")
    if variance > avg * 0.3:
        print("WARNING: Significant length variation detected - possible bias.")
    else:
        print("Length variation within acceptable range.")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Bias Detection Test ===")
    print("James Smith (American): 145 words")
    print("Maria Garcia (Mexican-American): 142 words")
    print("Wei Chen (Chinese-American): 148 words")
    print("Aisha Patel (Indian-American): 140 words")
    print(f"\nWord count range: 140-148 (variance: 8)")
    print("Length variation within acceptable range.")
    print("\nNote: Length is just one metric. A thorough bias audit would also check:")
    print("- Adjective choice (e.g., 'hardworking' vs 'brilliant')")
    print("- Recommendation strength ('competent' vs 'exceptional')")
    print("- Stereotypical associations")
    print("- Disaggregated performance metrics across demographics")

---
## 6. IAM Configuration for Vertex AI

Demonstrate the IAM roles and access patterns for AI workloads.

In [ ]:
# IAM role reference for Vertex AI
iam_roles = {
    "Developer": {
        "roles": ["roles/aiplatform.user"],
        "can_do": "Call model APIs, use Studio, run notebooks",
        "cannot_do": "Create/delete models, manage endpoints, admin settings"
    },
    "ML Engineer": {
        "roles": ["roles/aiplatform.user", "roles/aiplatform.modelUser"],
        "can_do": "Train models, deploy endpoints, run pipelines",
        "cannot_do": "Delete projects, manage IAM, billing changes"
    },
    "Platform Admin": {
        "roles": ["roles/aiplatform.admin"],
        "can_do": "Full Vertex AI management, configure security, manage resources",
        "cannot_do": "Should still not have project Owner (least privilege)"
    },
    "Auditor": {
        "roles": ["roles/aiplatform.viewer", "roles/logging.viewer"],
        "can_do": "View models, endpoints, logs (read-only)",
        "cannot_do": "Modify any resources, call generation APIs"
    },
    "Production App": {
        "roles": ["roles/aiplatform.user"],
        "can_do": "Call model APIs via service account",
        "cannot_do": "Human login, manage resources"
    },
}

print("=== IAM Roles for Vertex AI ===")
for persona, details in iam_roles.items():
    print(f"\n{persona}:")
    print(f"  Roles: {', '.join(details['roles'])}")
    print(f"  Can do: {details['can_do']}")
    print(f"  Cannot: {details['cannot_do']}")

print("\n" + "="*50)
print("Best Practices:")
print("  1. Use service accounts (not personal) for production apps")
print("  2. Apply least-privilege principle")
print("  3. Separate projects for dev/staging/prod")
print("  4. Enable Cloud Audit Logs for all Vertex AI operations")
print("  5. Use VPC Service Controls for data exfiltration prevention")

---
## 7. Implementation Readiness Assessment

Use Gemini to assess an organization's readiness for gen AI deployment.

In [ ]:
# Use Gemini as an AI readiness assessment tool
try:
    assessor = GenerativeModel(
        "gemini-1.5-pro",
        system_instruction="""You are an AI readiness consultant. Given a company profile,
assess their readiness for generative AI deployment on a scale of 1-5 in each category.
Provide specific, actionable recommendations. Be concise."""
    )

    company_profile = """Company: MidTech Corp (500 employees, $100M revenue)
- Industry: Financial services
- Current tech: AWS cloud, basic ML models for fraud detection
- Data: Customer data in PostgreSQL, documents in SharePoint
- Team: 5 data scientists, 2 ML engineers, no dedicated AI team
- Budget: $500K allocated for AI initiatives
- Goals: Automate customer support, improve document processing
- Concerns: Regulatory compliance (SOX, PCI-DSS), data privacy"""

    response = assessor.generate_content(
        f"Assess this company's readiness for Google Cloud generative AI:\n\n{company_profile}"
    )
    print("=== AI Readiness Assessment ===")
    print(response.text)

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Assessment ---")
    print("=== AI Readiness Assessment ===")
    print("")
    print("Data Readiness: 3/5")
    print("  Have structured data (PostgreSQL) but documents in SharePoint need")
    print("  migration to GCS for Vertex AI Search integration.")
    print("")
    print("Team Readiness: 2/5")
    print("  ML experience but no gen AI skills. Recommend: GCP AI training")
    print("  for 2-3 engineers + Google Cloud Skills Boost.")
    print("")
    print("Infrastructure: 2/5")
    print("  Currently on AWS. Need GCP migration plan or multi-cloud strategy.")
    print("  Consider starting with Gemini API (cloud-agnostic entry point).")
    print("")
    print("Governance: 4/5")
    print("  Strong compliance culture (SOX, PCI-DSS) is an advantage.")
    print("  Need to add AI-specific governance (SAIF framework).")
    print("")
    print("Budget: 3/5")
    print("  $500K is adequate for pilot phase. Recommend: $50K for Workspace")
    print("  (quick win), $200K for custom chatbot, $250K for infrastructure.")
    print("")
    print("Recommendation: Start with Gemini for Workspace (quick productivity win)")
    print("while building toward Vertex AI-powered document processing.")

---
## 8. Google's Responsible AI Principles Quiz

Test your knowledge of Google's AI principles for the exam.

In [ ]:
# Responsible AI Principles Quiz
rai_quiz = [
    {
        "q": "A model generates biased hiring recommendations. Which principle is violated?",
        "options": ["A) Transparency", "B) Fairness", "C) Privacy", "D) Accountability"],
        "answer": "B",
        "explanation": "Fairness: AI should avoid creating or reinforcing unfair bias, especially in consequential decisions like hiring."
    },
    {
        "q": "Users don't know they're chatting with an AI. Which principle is violated?",
        "options": ["A) Transparency", "B) Fairness", "C) Safety", "D) Social Benefit"],
        "answer": "A",
        "explanation": "Transparency: Users should know when they're interacting with AI systems."
    },
    {
        "q": "A model memorizes and outputs customer PII from training data. Which principle?",
        "options": ["A) Accountability", "B) Safety", "C) Privacy", "D) Transparency"],
        "answer": "C",
        "explanation": "Privacy: AI systems must protect personal information throughout the ML lifecycle."
    },
    {
        "q": "An AI system makes errors but nobody knows who is responsible. Which principle?",
        "options": ["A) Fairness", "B) Safety", "C) Transparency", "D) Accountability"],
        "answer": "D",
        "explanation": "Accountability: Clear ownership and responsibility must be established for AI systems."
    },
    {
        "q": "Which Google tool embeds watermarks in AI-generated content?",
        "options": ["A) DLP API", "B) SynthID", "C) Model Cards", "D) Safety Filters"],
        "answer": "B",
        "explanation": "SynthID embeds imperceptible watermarks in AI-generated images and text for transparency."
    },
    {
        "q": "Which framework does Google recommend for securing AI systems?",
        "options": ["A) NIST AI RMF", "B) ISO 42001", "C) SAIF", "D) SOC2"],
        "answer": "C",
        "explanation": "SAIF (Secure AI Framework) is Google's comprehensive framework for AI security."
    },
]

print("=== Responsible AI & Security Quiz ===")
for i, q in enumerate(rai_quiz):
    print(f"\nQ{i+1}: {q['q']}")
    for opt in q['options']:
        marker = " >>" if opt.startswith(q['answer']) else "   "
        print(f"{marker} {opt}")
    print(f"   Explanation: {q['explanation']}")

print("\n" + "="*50)
print("Google's 7 AI Principles:")
print("  1. Be socially beneficial")
print("  2. Avoid creating or reinforcing unfair bias")
print("  3. Be built and tested for safety")
print("  4. Be accountable to people")
print("  5. Incorporate privacy design principles")
print("  6. Uphold high standards of scientific excellence")
print("  7. Be made available for uses that accord with these principles")

---
## Summary

In this notebook, you covered:

1. **ROI Calculator** - Quantifying the business case for generative AI
2. **SAIF Assessment** - Evaluating security posture with Google's framework
3. **Safety Filters** - Configuring content safety for responsible deployment
4. **Bias Detection** - Testing model outputs for demographic bias
5. **IAM Configuration** - Proper access control for AI workloads
6. **Readiness Assessment** - Evaluating organizational preparedness
7. **Responsible AI Quiz** - Testing knowledge of Google's AI principles

**Key exam takeaways:**
- SAIF = Google's AI security framework (6 core elements)
- Safety filters are configurable per-request with 4 threshold levels
- Google's 7 AI principles (know all 7!)
- IAM: least privilege, service accounts for production, separate environments
- ROI metrics: productivity, cost reduction, revenue impact, quality
- Customer data in Vertex AI is NOT used to train Google's models
- SynthID for AI content watermarking

**Congratulations!** You've completed all 4 sections of the GCP Generative AI Leader study path.

**Back to**: [GAL Study Hub](../index.html)